In [ ]:
# Make sure to import packages and run code from the initial_setup notebook.
# SMAC will struggle if using Python versions newer than 3.11.

# The below code will run an RBFInterpolator surrogate model using Sequential Model-based Algorithm Configuration as an acquisition function.
# To test which kernel is the best for any particular function for RBFInterpolator, use the attached cross_validation_RBFInterpolator.ipynb. 

# Generating default random value for consistency.
rng = np.random.default_rng(19)

# In this example, we'll use Function 1. To run this analysis on a different function, simply swap it out.
# For example, insert df_function_5 instead of df_function_1.
data = df_function_1

# To use this code for other functions, add X values. For example, for Function 5, the X values would be:
# X_1 = df_cv['X_1'].values
# X_2 = df_cv['X_2'].values
# X_3 = df_cv['X_3'].values
# X_4 = df_cv['X_4'].values
X_1 = data['X_1'].values
X_2 = data['X_2'].values

# y stays the same for all functions.
y = data['y'].values

# Aggressive output scaling plus StandardScaler for Function 1. For Functions 2-8, I used the yeo-johnson power transformation below as recommended by HEBO. 
# When running Functions 2-8, it's recommended to comment out this aggressive scaling as it will negatively impact your results.
alpha = 0.02
y_eng = np.sign(y) * np.power(np.abs(y), alpha)

# scaler = StandardScaler()
y_eng = scaler.fit_transform(y_eng.reshape(-1, 1)).flatten()

# Scale y values for Functions 2-8. 
# pt = PowerTransformer(method='yeo-johnson')
# y_eng = pt.fit_transform(y.reshape(-1, 1))

# Add additional X values if running other functions, similar to the above. For example, running this on Function 5
# would look like points = np.column_stack((X_1, X_2, X_3, X_4))
points = np.column_stack((X_1, X_2))

model = RBFInterpolator(
    points,
    y_eng,
    kernel='gaussian',
    epsilon=16.0,
    smoothing=1e-6
)

# Define bounds using ConfigSpace. For functions with more dimensions, add X values. For example, running this on 
# Function 5 would look like this:
# configspace.add([
#     Float('X_1', bounds=(0.0, 0.999999)),
#     Float('X_2', bounds=(0.0, 0.999999)),
#     Float('X_3', bounds=(0.0, 0.999999)),
#     Float('X_4', bounds=(0.0, 0.999999))
# ])
configspace = ConfigurationSpace()
configspace.add([
    Float('X_1', bounds=(0.0, 0.999999)),
    Float('X_2', bounds=(0.0, 0.999999))
])

# Convert known points from DataFrame into Configuration objects. For functions with more dimensions, add X values.
# For example, running this on Function 5 would look like this:
# for x1, x2, x3, x4 in zip(data['X_1'], data['X_2'], data['X_3'], data['X_4']):
#     config = Configuration(
#         configspace, 
#         values={'X_1': float(x1), 'X_2': float(x2), 'X_3': float(x3), 'X_4': float(x4)}
#     )
#     initial_configs.append(config)
initial_configs = []
for x1, x2 in zip(data['X_1'], data['X_2']):
    config = Configuration(
        configspace, 
        values={'X_1': float(x1), 'X_2': float(x2)}
    )
    initial_configs.append(config)

# Define the objective function. For functions with more dimensions, add X values.
# For example, running this on Function 5 would look like this:
# def unknown_equation(config: Configuration, seed: int = 19) -> float:
#     x1 = config['X_1']
#     x2 = config['X_2']
#     x3 = config['X_3']
#     x4 = config['X_4']

#     y_return_array = model([[x1, x2, x3, x4]]) 
#     y_return = y_return_array[0, 0] 
def unknown_equation(config: Configuration, seed: int = 19) -> float:
    x1 = config['X_1']
    x2 = config['X_2']
    
    # Wrap inputs into a 2D array of shape (1, 2)
    y_return_array = model([[x1, x2]]) 
    # If running for Function 1, use the below.
    # If running for Functions 2-8, use y_return_array[0, 0].
    y_return = y_return_array[0] 
    # y_return = y_return_array[0, 0]
    
    # Return the negative target since SMAC minimizes
    return -float(y_return)

# Define the SMAC Scenario
scenario = Scenario(
    configspace,
    deterministic=True, 
    n_trials=250 # Increasing this will give you better results but can be very time-consuming
)

# Generate an initial design that includes your data points as additions
initial_design = HyperparameterOptimizationFacade.get_initial_design(
    scenario,
    additional_configs=initial_configs
)

# Initialize SMAC with the custom initial design
smac = HyperparameterOptimizationFacade(
    scenario, 
    unknown_equation,
    initial_design=initial_design,
    overwrite=True 
)

best_config = smac.optimize()

# Add X values to print if using higher-dimensional functions.
print(f"Best X_1 found: {best_config['X_1']:.6f}")
print(f"Best X_2 found: {best_config['X_2']:.6f}")

# Add X values if using higher dimensional functions.
best_point = np.array([[best_config["X_1"], best_config["X_2"]]])

# If running for Function 1, use the below. If running for
# Functions 2-8, use model(best_point)[0, 0].
predicted_y = model(best_point)[0]
# predicted_y = model(best_point)[0, 0]

# Inverse transforms for aggressive scaling in Function 1
pred_y_undo_scaler = scaler.inverse_transform([[predicted_y]])[0]
pred_y_orig_scale = np.sign(pred_y_undo_scaler) * np.power(np.abs(pred_y_undo_scaler), 1 / alpha)

# Inverse transforms for scaling in Functions 2-8
# pred_y_orig_scale = pt.inverse_transform([[predicted_y]])[0, 0]

print(f"Predicted objective (original scale): {pred_y_orig_scale}")

[INFO][abstract_initial_design.py:143] Using 20 initial design configurations and 23 additional configurations.
[INFO][abstract_intensifier.py:313] Using only one seed for deterministic scenario.
[INFO][abstract_intensifier.py:523] Added config 5b8d5e as new incumbent because there are no incumbents yet.
[INFO][abstract_intensifier.py:630] Added config 1a138c and rejected config 5b8d5e as incumbent because it is not better than the incumbents on 1 instances: 
[INFO][abstract_intensifier.py:630] Added config b5b759 and rejected config 1a138c as incumbent because it is not better than the incumbents on 1 instances: 
[INFO][abstract_intensifier.py:630] Added config 9f461e and rejected config b5b759 as incumbent because it is not better than the incumbents on 1 instances: 
[INFO][abstract_intensifier.py:630] Added config e2b9cc and rejected config 9f461e as incumbent because it is not better than the incumbents on 1 instances: 
[INFO][abstract_intensifier.py:630] Added config bc8e23 and re